# LQ TinyClassifier —— 训练 Notebook（程序化调用）

直接 import 脚本内部函数，分步执行训练全过程。

支持两种 backbone：
- `tiny_custom` — 原深度可分离卷积架构（4K 参数），产物输出到 `LQ_TinyClassifier/artifacts/`
- `mobilenet_v2` — MobileNetV2（2.2M 参数），产物输出到 `LQ_TinyClassifier_MobileNetV2/artifacts_mobilenetv2/`

**不修改原始代码的任何文件。**

数据集需为 ImageFolder 结构：
```
dataset/
  class_a/
    xxx.jpg
  class_b/
    yyy.jpg
```

In [ ]:
# 1. 导入依赖
import sys
import json
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torchvision import datasets, transforms

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# 2. 检查数据集（修改 DATA_ROOT 为你的路径）
DATA_ROOT = "../dataset/step3_5.OpenCV-局部光照"  # 替换为你的数据集路径
data_root = Path(DATA_ROOT)

if not data_root.exists() or not data_root.is_dir():
    raise FileNotFoundError(f"数据集目录不存在: {data_root}")

class_dirs = sorted([p.name for p in data_root.iterdir() if p.is_dir()])
print(f"类别数: {len(class_dirs)}")
print(f"类别列表: {class_dirs}")

for cls_name in class_dirs:
    imgs = list((data_root / cls_name).iterdir())
    print(f"  {cls_name}: {len(imgs)} 张图片")

In [ ]:
# 3. 导入训练函数 + 参数配置 + 加载数据
# 从 LQ_TinyClassifier 导入原始模型的公共函数（不修改原始文件）
sys.path.insert(0, str(Path.cwd() / "LQ_TinyClassifier"))
# 从 LQ_TinyClassifier_MobileNetV2 导入 MobileNetV2 模型类
sys.path.insert(0, str(Path.cwd() / "LQ_TinyClassifier_MobileNetV2"))

from train_tiny_classifier import (
    TinyClassifier,
    set_seed,
    build_splits,
    get_data_loaders,
    run_epoch,
    export_onnx,
    benchmark_torch_cpu,
)
from train_mobilenet_v2 import MobileNetV2Classifier

# ===== 训练参数配置 =====
# 修改 backbone 字段切换主干网络：
#   "tiny_custom"  — 原深度可分离卷积架构（4K 参数），产物 -> LQ_TinyClassifier/artifacts/
#   "mobilenet_v2" — MobileNetV2（2.2M 参数），产物 -> LQ_TinyClassifier_MobileNetV2/artifacts_mobilenetv2/
CFG = {
    "data_root":     DATA_ROOT,
    "img_size":      96,
    "batch_size":    64,
    "epochs":        50,
    "lr":            5e-5,
    "weight_decay":  1e-4,
    "num_workers":   4,
    "seed":          42,
    "width_mult":    0.35,          # 可在此修改
    "backbone":      "mobilenet_v2", # ← 在这里切换 backbone
    "dropout":        0.6,              # 分类头 Dropout 概率，0.0 表示不启用（仅 mobilenet_v2 生效）
    "patience":      10,
    "target_acc":    0.98,
}
# =========================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"设备: {device}")

set_seed(CFG["seed"])
train_loader, val_loader, test_loader, classes, class_to_idx = get_data_loaders(
    data_root=Path(CFG["data_root"]),
    img_size=CFG["img_size"],
    batch_size=CFG["batch_size"],
    num_workers=CFG["num_workers"],
    seed=CFG["seed"],
)
num_classes = len(classes)
print(f"类别: {classes}")
print(f"训练集: {len(train_loader.dataset)}")
print(f"验证集: {len(val_loader.dataset)}")
print(f"测试集: {len(test_loader.dataset)}")

In [ ]:
# 4. 根据 backbone 选择对应模型类 + 设置输出路径
backbone = CFG["backbone"]
if backbone == "tiny_custom":
    ModelClass = TinyClassifier
    CFG.setdefault("width_mult", 0.35)
    CFG["out_dir"] = "LQ_TinyClassifier/artifacts"
    onnx_name = "tiny_classifier_fp32.onnx"
    model_kwargs = {}
elif backbone == "mobilenet_v2":
    ModelClass = MobileNetV2Classifier
    CFG.setdefault("width_mult", 1.0)
    CFG["out_dir"] = "LQ_TinyClassifier_MobileNetV2/artifacts_mobilenetv2"
    onnx_name = "mobilenet_v2_fp32.onnx"
    model_kwargs = {"dropout": CFG.get("dropout", 0.0)}
else:
    raise ValueError(f"Unknown backbone: {backbone}")

out_dir = Path(CFG["out_dir"])
out_dir.mkdir(parents=True, exist_ok=True)

model = ModelClass(
    num_classes=num_classes,
    width_mult=CFG["width_mult"],
    **model_kwargs,
).to(device)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_params = sum(p.numel() for p in model.parameters())
print(f"Backbone: {backbone}")
print(f"输出目录: {out_dir}")
print(f"模型参数量: {n_params}（可训练: {trainable_params}）")
print(f"分类头 Dropout: {CFG.get('dropout', 0.0)}")
print(model)

In [ ]:
# 5. 训练循环（保存时记录 backbone 名称）
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG["epochs"])

best_val_acc = 0.0
best_epoch = -1
best_path = out_dir / "best_model.pt"
bad_epochs = 0
reached_target_acc = False

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(1, CFG["epochs"] + 1):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = run_epoch(model, val_loader, criterion, None, device)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch:02d}/{CFG['epochs']} | "
          f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
          f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch
        bad_epochs = 0
        torch.save(
            {
                "model_state": model.state_dict(),
                "classes": classes,
                "class_to_idx": class_to_idx,
                "img_size": CFG["img_size"],
                "width_mult": CFG["width_mult"],
                "backbone": backbone,         # ← 保存 backbone 名称
                "mean": [0.485, 0.456, 0.406],
                "std": [0.229, 0.224, 0.225],
            },
            best_path,
        )
    else:
        bad_epochs += 1

    if best_val_acc >= CFG["target_acc"]:
        reached_target_acc = True
        print(f"达到目标准确率 {CFG['target_acc']:.4f}，提前停止训练。")
        break

    if bad_epochs >= CFG["patience"]:
        print(f"早停于 epoch {epoch}，最佳 epoch = {best_epoch}")
        break

print(f"\n最佳验证准确率: {best_val_acc:.4f} @ epoch {best_epoch}")

In [ ]:
# 6. 测试集评估（加载最佳 checkpoint 后评估）
checkpoint = torch.load(best_path, map_location="cpu")
model.load_state_dict(checkpoint["model_state"])
model = model.to(device)
test_loss, test_acc = run_epoch(model, test_loader, criterion, None, device)
print(f"测试集损失: {test_loss:.4f}")
print(f"测试集准确率: {test_acc:.4f}")

In [ ]:
# 7. 导出 ONNX + CPU 测速
model_cpu = model.cpu().eval()
onnx_path = out_dir / onnx_name
export_onnx(model_cpu, onnx_path, CFG["img_size"])
print(f"ONNX 导出完成: {onnx_path}")

cpu_ms = benchmark_torch_cpu(model_cpu, CFG["img_size"])
print(f"CPU 单线程推理延迟: {cpu_ms:.3f} ms")

In [ ]:
# 8. 保存训练指标（含 backbone 字段）
metrics = {
    "best_val_acc": best_val_acc,
    "best_epoch": best_epoch,
    "target_acc": CFG["target_acc"],
    "reached_target_acc": reached_target_acc,
    "test_acc": test_acc,
    "test_loss": test_loss,
    "torch_cpu_1thread_ms": cpu_ms,
    "num_params": n_params,
    "backbone": backbone,
    "classes": classes,
    "img_size": CFG["img_size"],
    "width_mult": CFG["width_mult"],
}

labels_path = out_dir / "labels.txt"
with open(labels_path, "w", encoding="utf-8") as f:
    for name in classes:
        f.write(f"{name}\n")

metrics["labels_path"] = str(labels_path)
metrics["onnx_path"] = str(onnx_path)

with open(out_dir / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print(f"保存完成：")
print(f"  模型: {best_path}")
print(f"  ONNX: {onnx_path}")
print(f"  标签: {labels_path}")
print(f"  指标: {out_dir / 'metrics.json'}")

In [ ]:
# 9. 绘制训练曲线
import matplotlib.pyplot as plt

epochs_range = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_range, history["train_loss"], label="Train Loss", marker="o")
axes[0].plot(epochs_range, history["val_loss"], label="Val Loss", marker="s")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss Curve")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history["train_acc"], label="Train Acc", marker="o")
axes[1].plot(epochs_range, history["val_acc"], label="Val Acc", marker="s")
axes[1].axhline(y=CFG["target_acc"], color="r", linestyle="--", alpha=0.5, label="Target")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy Curve")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 10. 训练结果摘要（打印 backbone 信息）
print("===== 训练结果摘要 =====")
print(f"Backbone: {backbone}")
print(f"输出目录: {out_dir}")
print(f"类别: {classes}")
print(f"参数量: {n_params}")
print(f"最佳验证准确率: {best_val_acc:.4f} (epoch {best_epoch})")
print(f"目标准确率: {CFG['target_acc']} | 已达标: {reached_target_acc}")
print(f"测试集准确率: {test_acc:.4f}")
print(f"CPU 推理延迟: {cpu_ms:.3f} ms")

---
## 🔍 推理 Demo

用训练好的模型对单张图片进行预测。

In [ ]:
# 11. 加载 checkpoint 并创建推理模型（从 checkpoint 中读取 backbone）
from PIL import Image

def predict(model, image_path, img_size, classes, mean, std):
    model.eval()
    img = Image.open(image_path).convert("RGB")
    tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    x = tf(img).unsqueeze(0)
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1).squeeze(0)
        pred_idx = logits.argmax(dim=1).item()

    print(f"图片: {image_path}")
    print(f"预测类别: {classes[pred_idx]} (索引 {pred_idx})")
    print(f"置信度分布:")
    for i, cls_name in enumerate(classes):
        print(f"  {cls_name}: {probs[i].item():.4f}")
    return classes[pred_idx], probs[pred_idx].item()


checkpoint_path = out_dir / "best_model.pt"
ckpt = torch.load(checkpoint_path, map_location="cpu")
cls_names = ckpt["classes"]
wm = ckpt.get("width_mult", 0.6)
backbone_from_ckpt = ckpt.get("backbone", "tiny_custom")  # ← 读取 backbone
im_size = ckpt.get("img_size", 96)
mean = ckpt.get("mean", [0.485, 0.456, 0.406])
std = ckpt.get("std", [0.229, 0.224, 0.225])

# 根据 checkpoint 中的 backbone 选择对应模型类
if backbone_from_ckpt == "tiny_custom":
    InferModelClass = TinyClassifier
elif backbone_from_ckpt == "mobilenet_v2":
    InferModelClass = MobileNetV2Classifier
else:
    raise ValueError(f"Unknown backbone from checkpoint: {backbone_from_ckpt}")

infer_model = InferModelClass(
    num_classes=len(cls_names),
    width_mult=wm,
)
infer_model.load_state_dict(ckpt["model_state"])

print("模型加载成功！")
print(f"Backbone: {backbone_from_ckpt}")
print(f"类别: {cls_names}")
print(f"输入尺寸: {im_size}x{im_size}")

In [ ]:
# 12. 自动选一张验证集图片做推理演示
test_root = Path(CFG["data_root"])
class_dirs_avail = [d for d in test_root.iterdir() if d.is_dir()]
if class_dirs_avail:
    first_class = class_dirs_avail[0]
    images = list(first_class.glob("*"))[:1]
    if images:
        predict(infer_model, str(images[0]), im_size, cls_names, mean, std)
    else:
        print("第一个类别目录下没有图片。")
else:
    print(f"{test_root} 下没有类别目录。")

---
## 附录：训练产物说明

训练产物根据 `backbone` 输出到各自文件夹：

### tiny_custom → `LQ_TinyClassifier/artifacts/`
| 文件 | 说明 |
|------|------|
| `best_model.pt` | 最佳模型 checkpoint（权重 + 元信息） |
| `tiny_classifier_fp32.onnx` | ONNX 格式模型 |
| `labels.txt` | 类别标签（每行一个） |
| `metrics.json` | 训练指标 |

### mobilenet_v2 → `LQ_TinyClassifier_MobileNetV2/artifacts_mobilenetv2/`
| 文件 | 说明 |
|------|------|
| `best_model.pt` | 最佳模型 checkpoint（权重 + 元信息） |
| `mobilenet_v2_fp32.onnx` | ONNX 格式模型 |
| `labels.txt` | 类别标签（每行一个） |
| `metrics.json` | 训练指标 |